
# Notebook 5 - Evaluation + BigQuery Logging (Colab + uv)

This notebook evaluates the agentic address intelligence system and writes evaluation results to BigQuery.

Locked architecture stays the same:
1. Coordinator Agent
2. Retrieval Agent
3. Address Standardization Agent
4. Commute & Confidence Agent
5. Eligibility Agent
6. Escalation Agent

Notebook 5 focuses on:
- evaluating the Address Standardization Agent
- evaluating the full Coordinator workflow
- comparing base model vs fine-tuned adapter
- computing automated metrics
- preparing A/B-style comparisons
- logging evaluation outputs to BigQuery



## Step 0: install uv


In [ ]:

!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] += ":/root/.local/bin"
!uv --version



## Step 1: install packages with uv


In [ ]:

import os
os.environ["PATH"] += ":/root/.local/bin"

#!uv pip install --system   numpy==1.26.4   pandas   datasets   transformers   accelerate   bitsandbytes   peft   sentence-transformers   langchain   langchain-community   langchain-huggingface   faiss-cpu   google-cloud-bigquery   db-dtypes   scikit-learn   requests==2.32.4

!pip uninstall -y numpy pandas faiss-cpu sentence-transformers transformers accelerate bitsandbytes langchain langchain-core langchain-community langchain-huggingface
!pip uninstall -y trl transformers peft accelerate bitsandbytes
!pip install --no-cache-dir \
  numpy==1.26.4 \
  pandas==2.2.2 \
  langchain \
  langchain-community \
  langchain-huggingface \
  faiss-cpu \
  sentence-transformers \
  transformers \
  accelerate \
  bitsandbytes \
  google-cloud-bigquery \
  db-dtypes \
  requests==2.32.4

In [ ]:
!pip uninstall -y trl transformers peft accelerate bitsandbytes
!pip install -U "trl[peft]" bitsandbytes datasets wandb


## Step 2: optional file upload

Optional files you may upload into Colab:
- `multi_agent_helpers_uv.py`
- `finetuning_helpers_uv.py`

This notebook is self-contained, so upload is optional.


In [ ]:

from google.colab import files
import os

print("Current files before optional upload:")
print(os.listdir())

import transformers, trl, peft, accelerate
print("transformers:", transformers.__version__)
print("trl:", trl.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)

from peft import get_peft_model, LoraConfig, prepare_model_for_kbit_training
print("PEFT import OK")

import numpy as np
import pandas as pd
import torch

print(np.__version__)
print(pd.__version__)
print(torch.cuda.is_available())

# OPTIONAL:
# uploaded = files.upload()



## Step 3: imports


In [ ]:

import os
import json
import re
import torch
import pandas as pd
from difflib import SequenceMatcher

from google.colab import auth
from google.cloud import bigquery

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from sentence_transformers import CrossEncoder
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline
)
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from peft import PeftModel



## Step 4: authenticate and connect to BigQuery


In [ ]:

auth.authenticate_user()
print("Authenticated successfully")

PROJECT_ID = "" #add your project id here
DATASET_ID = "address_intelligence_demo" #add your database id here
TABLE_EMPLOYEES = "employee_input"
TABLE_KB = "kb_documents"
TABLE_OUTPUT = "agent_output_predictions"

client = bigquery.Client(project=PROJECT_ID)
print("Connected to BigQuery project:", PROJECT_ID)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))



## Step 5: read employee + KB data from BigQuery


In [ ]:

kb_query = f'''
SELECT doc_id, source_type, title, text
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_KB}`
ORDER BY doc_id
'''

employee_query = f'''
SELECT employee_id, employee_name, office_id, raw_home_address, office_address
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_EMPLOYEES}`
ORDER BY employee_id
'''

kb_df = client.query(kb_query).to_dataframe()
employee_df = client.query(employee_query).to_dataframe()

print("KB rows:", len(kb_df))
print("Employee rows:", len(employee_df))
employee_df.head()



## Step 6: build evaluation ground truth

We create a small labeled evaluation set.


In [ ]:

ground_truth_records = [
    {
        "employee_id": "E001",
        "expected_corrected_address": "123 Main St Apt 2, Nashville, TN 37203",
        "expected_decision": "within_60_miles",
        "expected_manual_review": False
    },
    {
        "employee_id": "E003",
        "expected_corrected_address": "789 Broadway, Nashville, TN 37203",
        "expected_decision": "within_60_miles",
        "expected_manual_review": False
    },
    {
        "employee_id": "E010",
        "expected_corrected_address": "404 Not Found Ln, Brentwood, TN 37027",
        "expected_decision": "within_60_miles",
        "expected_manual_review": True
    },
    {
        "employee_id": "E017",
        "expected_corrected_address": "1 Apple Park Way, Cupertino, CA 95014",
        "expected_decision": "outside_60_miles",
        "expected_manual_review": False
    },
    {
        "employee_id": "E019",
        "expected_corrected_address": "1818 Church St, Nashville, TN 37203",
        "expected_decision": "within_60_miles",
        "expected_manual_review": False
    }
]

ground_truth_df = pd.DataFrame(ground_truth_records)
ground_truth_df



## Step 7: rebuild retrieval stack from earlier notebooks


In [ ]:

documents = []
for _, row in kb_df.iterrows():
    documents.append(
        Document(
            page_content=row["text"],
            metadata={
                "doc_id": row["doc_id"],
                "source_type": row["source_type"],
                "title": row["title"],
            }
        )
    )

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)
vectorstore = FAISS.from_documents(documents, embeddings)

RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
reranker = CrossEncoder(RERANKER_MODEL_NAME)

print("Retrieval stack ready")


In [ ]:
import os
print(os.listdir())

from google.colab import files
#uploaded = files.upload()
!unzip '/content/address_agent_qlora_adapter.zip'
!ls address_agent_qlora_adapter

In [ ]:
print(os.listdir())


## Step 8: load base model and fine-tuned adapter model

We compare:
- base model
- fine-tuned adapter model


In [ ]:

BASE_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
ADAPTER_DIR = "address_agent_qlora_adapter"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model_for_eval = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=bnb_config if torch.cuda.is_available() else None,
    device_map="auto",
    trust_remote_code=True
)

base_pipe = pipeline(
    "text-generation",
    model=base_model_for_eval,
    tokenizer=tokenizer,
    max_new_tokens=220,
    do_sample=False,
    return_full_text=True
)

ft_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=bnb_config if torch.cuda.is_available() else None,
    device_map="auto",
    trust_remote_code=True
)
ft_model = PeftModel.from_pretrained(ft_base, ADAPTER_DIR)

ft_pipe = pipeline(
    "text-generation",
    model=ft_model,
    tokenizer=tokenizer,
    max_new_tokens=220,
    do_sample=False,
    return_full_text=True
)

print("Base and fine-tuned pipelines ready")



## Step 9: helper functions from previous notebooks


In [ ]:

def run_chat_prompt(system_prompt: str, user_prompt: str, text_generation_pipeline, tokenizer) -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    output = text_generation_pipeline(prompt)[0]["generated_text"]
    answer = output[len(prompt):].strip()
    return answer

def safe_json_loads(text: str):
    text = text.strip()
    match = re.search(r'\{.*\}', text, flags=re.DOTALL)
    if match:
        text = match.group(0)
    return json.loads(text)

SYSTEM_PROMPT_QUERY_REWRITER = """
You are a retrieval query rewriting assistant for an enterprise address intelligence system.

Your job:
1. Read a messy employee-entered home address and an assigned office address.
2. Rewrite them into a clean, retrieval-friendly query.
3. Preserve important location details.
4. Expand the retrieval intent so the retriever can find:
   - commute eligibility policy
   - address standardization rules
   - typo correction examples
   - office metadata
   - manual review / escalation rules
5. Do not invent unsupported address fields.
6. Return only the rewritten retrieval query text.
"""

def build_user_prompt_for_rewriter(raw_home_address: str, office_address: str) -> str:
    return f"""
Raw employee home address:
{raw_home_address}

Assigned office address:
{office_address}

Rewrite this into a retrieval query that will help search the enterprise KB for the best supporting documents.
"""

def rewrite_query_with_hf_llm(raw_home_address: str, office_address: str, text_generation_pipeline, tokenizer) -> str:
    return run_chat_prompt(
        SYSTEM_PROMPT_QUERY_REWRITER,
        build_user_prompt_for_rewriter(raw_home_address, office_address),
        text_generation_pipeline,
        tokenizer
    )

def rerank_documents(query_text, docs, reranker_model, top_k=4):
    pairs = [[query_text, d.page_content] for d in docs]
    scores = reranker_model.predict(pairs)
    scored = list(zip(docs, scores))
    scored = sorted(scored, key=lambda x: x[1], reverse=True)

    final_docs = []
    for doc, score in scored[:top_k]:
        doc.metadata["rerank_score"] = float(score)
        final_docs.append(doc)
    return final_docs

def retrieval_agent(raw_home_address, office_address, vectorstore, text_generation_pipeline, tokenizer, reranker_model, candidate_k=8, top_k=4):
    rewritten_query = rewrite_query_with_hf_llm(
        raw_home_address=raw_home_address,
        office_address=office_address,
        text_generation_pipeline=text_generation_pipeline,
        tokenizer=tokenizer
    )
    candidate_docs = vectorstore.similarity_search(rewritten_query, k=candidate_k)
    final_docs = rerank_documents(rewritten_query, candidate_docs, reranker_model, top_k=top_k)
    return {
        "rewritten_query": rewritten_query,
        "retrieved_context": [
            {
                "doc_id": d.metadata["doc_id"],
                "title": d.metadata["title"],
                "source_type": d.metadata["source_type"],
                "rerank_score": d.metadata.get("rerank_score"),
                "text": d.page_content
            }
            for d in final_docs
        ]
    }

SYSTEM_PROMPT_ADDRESS_AGENT = """
You are an address standardization agent for an enterprise address intelligence system.

Use the retrieved policy, typo examples, and standardization rules to correct and standardize the employee address.

Rules:
1. Preserve valid information.
2. Correct obvious typos if strongly supported by evidence.
3. Do not invent unsupported address fields.
4. If an important field is missing, keep it missing and lower confidence.
5. Return valid JSON only.

Required JSON keys:
corrected_standardized_address
correction_confidence
standardization_reason
"""

def build_user_prompt_for_address_agent(raw_home_address: str, office_address: str, retrieved_context: list) -> str:
    return json.dumps({
        "raw_home_address": raw_home_address,
        "office_address": office_address,
        "retrieved_context": retrieved_context
    }, indent=2)

def address_standardization_agent(raw_home_address, office_address, retrieved_context, text_generation_pipeline, tokenizer):
    output = run_chat_prompt(
        SYSTEM_PROMPT_ADDRESS_AGENT,
        build_user_prompt_for_address_agent(raw_home_address, office_address, retrieved_context),
        text_generation_pipeline,
        tokenizer
    )
    try:
        data = safe_json_loads(output)
    except Exception:
        data = {
            "corrected_standardized_address": raw_home_address,
            "correction_confidence": 0.5,
            "standardization_reason": "Fallback parsing used because model output was not valid JSON."
        }
    return data

def heuristic_commute_estimate(home_address: str, office_address: str):
    text = home_address.lower()
    if "cupertino" in text or "california" in text:
        return 2100.0, 2100
    if "new york" in text:
        return 900.0, 900
    if "atlanta" in text:
        return 250.0, 250
    if "springfield" in text or "illinois" in text:
        return 430.0, 430
    if "franklin" in text or "brentwood" in text:
        return 20.0, 30
    if "nashville" in text:
        return 8.0, 20
    return 35.0, 45

def commute_and_confidence_agent(corrected_standardized_address, office_address, correction_confidence, retrieved_context):
    distance_miles, commute_minutes = heuristic_commute_estimate(corrected_standardized_address, office_address)

    rerank_scores = [x.get("rerank_score", 0.0) or 0.0 for x in retrieved_context]
    avg_rerank = sum(rerank_scores) / len(rerank_scores) if rerank_scores else 0.0
    avg_rerank = min(max(avg_rerank, 0.0), 1.0)

    geocode_match_quality = "high" if correction_confidence >= 0.85 else "medium" if correction_confidence >= 0.65 else "low"

    combined_confidence = min(
        0.99,
        max(
            0.05,
            0.85 * float(correction_confidence) + 0.15 * avg_rerank
        )
    )

    confidence_reasoning = (
        f"Combined confidence uses correction confidence={correction_confidence} "
        f"and average retrieval/rerank support={round(avg_rerank, 4)}."
    )

    manual_review_trigger_signals = []
    if correction_confidence < 0.75:
        manual_review_trigger_signals.append("low_address_correction_confidence")
    if geocode_match_quality == "low":
        manual_review_trigger_signals.append("low_geocode_match_quality")
    if distance_miles is None:
        manual_review_trigger_signals.append("missing_commute_distance")

    return {
        "travel_distance_miles": float(distance_miles) if distance_miles is not None else None,
        "estimated_commute_minutes": int(commute_minutes) if commute_minutes is not None else None,
        "geocode_match_quality": geocode_match_quality,
        "combined_confidence_score": round(float(combined_confidence), 4),
        "confidence_reasoning": confidence_reasoning,
        "manual_review_trigger_signals": manual_review_trigger_signals
    }

def eligibility_agent(travel_distance_miles, combined_confidence_score, correction_confidence):
    if travel_distance_miles is None:
        return {
            "eligibility_decision": "manual_review",
            "eligibility_reason": "Distance could not be computed."
        }

    if travel_distance_miles <= 60 and correction_confidence >= 0.90:
        return {
            "eligibility_decision": "within_60_miles",
            "eligibility_reason": "Distance is within 60 miles and address confidence is very high."
        }

    if travel_distance_miles <= 60 and combined_confidence_score >= 0.75:
        return {
            "eligibility_decision": "within_60_miles",
            "eligibility_reason": "Distance is within 60 miles and confidence is above threshold."
        }

    if travel_distance_miles > 60:
        return {
            "eligibility_decision": "outside_60_miles",
            "eligibility_reason": "Distance is greater than 60 miles."
        }

    return {
        "eligibility_decision": "manual_review",
        "eligibility_reason": "Distance is within threshold but confidence is not high enough."
    }

def escalation_agent(correction_confidence, manual_review_trigger_signals, eligibility_decision, corrected_standardized_address):
    reasons = list(manual_review_trigger_signals)

    if eligibility_decision == "manual_review":
        reasons.append("eligibility_rule_requires_review")

    if "," not in corrected_standardized_address:
        reasons.append("address_format_still_looks_incomplete")

    manual_review_required = len(reasons) > 0

    return {
        "manual_review_required": manual_review_required,
        "escalation_reason": "; ".join(reasons) if reasons else "no_escalation_needed"
    }

def coordinator_agent(employee_row, vectorstore, text_generation_pipeline, tokenizer, reranker_model, model_version):
    raw_home_address = employee_row["raw_home_address"]
    office_address = employee_row["office_address"]

    retrieval_output = retrieval_agent(
        raw_home_address=raw_home_address,
        office_address=office_address,
        vectorstore=vectorstore,
        text_generation_pipeline=text_generation_pipeline,
        tokenizer=tokenizer,
        reranker_model=reranker_model,
        candidate_k=8,
        top_k=4
    )

    address_output = address_standardization_agent(
        raw_home_address=raw_home_address,
        office_address=office_address,
        retrieved_context=retrieval_output["retrieved_context"],
        text_generation_pipeline=text_generation_pipeline,
        tokenizer=tokenizer
    )

    commute_output = commute_and_confidence_agent(
        corrected_standardized_address=address_output["corrected_standardized_address"],
        office_address=office_address,
        correction_confidence=address_output["correction_confidence"],
        retrieved_context=retrieval_output["retrieved_context"]
    )

    eligibility_output = eligibility_agent(
        travel_distance_miles=commute_output["travel_distance_miles"],
        combined_confidence_score=commute_output["combined_confidence_score"],
        correction_confidence=address_output["correction_confidence"]
    )

    escalation_output = escalation_agent(
        correction_confidence=address_output["correction_confidence"],
        manual_review_trigger_signals=commute_output["manual_review_trigger_signals"],
        eligibility_decision=eligibility_output["eligibility_decision"],
        corrected_standardized_address=address_output["corrected_standardized_address"]
    )

    final_output = {
        "employee_id": employee_row["employee_id"],
        "employee_name": employee_row["employee_name"],
        "raw_home_address": raw_home_address,
        "office_address": office_address,
        "corrected_standardized_address": address_output["corrected_standardized_address"],
        "correction_confidence": address_output["correction_confidence"],
        "travel_distance_miles": commute_output["travel_distance_miles"],
        "estimated_commute_minutes": commute_output["estimated_commute_minutes"],
        "eligibility_decision": eligibility_output["eligibility_decision"],
        "manual_review_required": escalation_output["manual_review_required"],
        "escalation_reason": escalation_output["escalation_reason"],
        "prompt_version": "v2_notebook5_eval",
        "model_version": model_version,
        "retrieval_context_titles": [x["title"] for x in retrieval_output["retrieved_context"]],
        "rewritten_query": retrieval_output["rewritten_query"],
        "confidence_reasoning": commute_output["confidence_reasoning"],
        "eligibility_reason": eligibility_output["eligibility_reason"],
        "standardization_reason": address_output["standardization_reason"]
    }
    return final_output



## Step 10: run evaluation for base model and fine-tuned model


In [ ]:

eval_rows = []
eval_employee_df = employee_df.merge(ground_truth_df, on="employee_id", how="inner")

for _, row in eval_employee_df.iterrows():
    base_result = coordinator_agent(
        employee_row=row,
        vectorstore=vectorstore,
        text_generation_pipeline=base_pipe,
        tokenizer=tokenizer,
        reranker_model=reranker,
        model_version="base_qwen"
    )
    ft_result = coordinator_agent(
        employee_row=row,
        vectorstore=vectorstore,
        text_generation_pipeline=ft_pipe,
        tokenizer=tokenizer,
        reranker_model=reranker,
        model_version="ft_qwen_lora"
    )

    eval_rows.append({
        "employee_id": row["employee_id"],
        "employee_name": row["employee_name"],
        "expected_corrected_address": row["expected_corrected_address"],
        "expected_decision": row["expected_decision"],
        "expected_manual_review": row["expected_manual_review"],

        "base_corrected_address": base_result["corrected_standardized_address"],
        "base_correction_confidence": base_result["correction_confidence"],
        "base_decision": base_result["eligibility_decision"],
        "base_manual_review": base_result["manual_review_required"],

        "ft_corrected_address": ft_result["corrected_standardized_address"],
        "ft_correction_confidence": ft_result["correction_confidence"],
        "ft_decision": ft_result["eligibility_decision"],
        "ft_manual_review": ft_result["manual_review_required"],

        "base_output_json": json.dumps(base_result),
        "ft_output_json": json.dumps(ft_result),
    })

eval_results_df = pd.DataFrame(eval_rows)
eval_results_df



## Step 11: automated metrics


In [ ]:

def string_similarity(a: str, b: str) -> float:
    return SequenceMatcher(None, str(a), str(b)).ratio()

eval_results_df["base_address_similarity"] = eval_results_df.apply(
    lambda x: string_similarity(x["base_corrected_address"], x["expected_corrected_address"]), axis=1
)
eval_results_df["ft_address_similarity"] = eval_results_df.apply(
    lambda x: string_similarity(x["ft_corrected_address"], x["expected_corrected_address"]), axis=1
)

eval_results_df["base_decision_correct"] = (eval_results_df["base_decision"] == eval_results_df["expected_decision"]).astype(int)
eval_results_df["ft_decision_correct"] = (eval_results_df["ft_decision"] == eval_results_df["expected_decision"]).astype(int)

eval_results_df["base_manual_review_correct"] = (eval_results_df["base_manual_review"] == eval_results_df["expected_manual_review"]).astype(int)
eval_results_df["ft_manual_review_correct"] = (eval_results_df["ft_manual_review"] == eval_results_df["expected_manual_review"]).astype(int)

summary_metrics = {
    "base_avg_address_similarity": round(eval_results_df["base_address_similarity"].mean(), 4),
    "ft_avg_address_similarity": round(eval_results_df["ft_address_similarity"].mean(), 4),
    "base_decision_accuracy": round(eval_results_df["base_decision_correct"].mean(), 4),
    "ft_decision_accuracy": round(eval_results_df["ft_decision_correct"].mean(), 4),
    "base_manual_review_accuracy": round(eval_results_df["base_manual_review_correct"].mean(), 4),
    "ft_manual_review_accuracy": round(eval_results_df["ft_manual_review_correct"].mean(), 4),
    "base_avg_confidence": round(eval_results_df["base_correction_confidence"].mean(), 4),
    "ft_avg_confidence": round(eval_results_df["ft_correction_confidence"].mean(), 4),
}

summary_metrics



## Step 12: build a readable comparison table


In [ ]:

comparison_df = eval_results_df[[
    "employee_id",
    "employee_name",
    "expected_corrected_address",
    "base_corrected_address",
    "ft_corrected_address",
    "base_address_similarity",
    "ft_address_similarity",
    "expected_decision",
    "base_decision",
    "ft_decision",
    "expected_manual_review",
    "base_manual_review",
    "ft_manual_review",
]]
comparison_df



## Step 13: identify wins / losses for the fine-tuned model


In [ ]:

def compare_row(row):
    base_score = row["base_address_similarity"] + row["base_decision_correct"] + row["base_manual_review_correct"]
    ft_score = row["ft_address_similarity"] + row["ft_decision_correct"] + row["ft_manual_review_correct"]
    if ft_score > base_score:
        return "ft_better"
    if ft_score < base_score:
        return "base_better"
    return "tie"

eval_results_df["ab_outcome"] = eval_results_df.apply(compare_row, axis=1)
eval_results_df["ab_outcome"].value_counts()



## Step 14: prepare BigQuery evaluation tables


In [ ]:

row_level_eval_table = "agent_eval_row_level"
summary_eval_table = "agent_eval_summary"

row_level_bq_df = eval_results_df.copy()
row_level_bq_df["run_timestamp"] = pd.Timestamp.utcnow()

summary_bq_df = pd.DataFrame([summary_metrics])
summary_bq_df["run_timestamp"] = pd.Timestamp.utcnow()
summary_bq_df["evaluation_name"] = "notebook5_base_vs_ft_ab"
summary_bq_df["base_model"] = BASE_MODEL_NAME
summary_bq_df["ft_model"] = ADAPTER_DIR

print("Prepared evaluation dataframes")



## Step 15: write evaluation results to BigQuery


In [ ]:

row_table_id = f"{PROJECT_ID}.{DATASET_ID}.{row_level_eval_table}"
summary_table_id = f"{PROJECT_ID}.{DATASET_ID}.{summary_eval_table}"

job1 = client.load_table_from_dataframe(row_level_bq_df, row_table_id)
job1.result()

job2 = client.load_table_from_dataframe(summary_bq_df, summary_table_id)
job2.result()

print("Wrote row-level evaluation table:", row_table_id)
print("Wrote summary evaluation table:", summary_table_id)
print("Row count:", len(row_level_bq_df))



## Step 16: save local artifacts


In [ ]:

with open("notebook5_summary_metrics.json", "w") as f:
    json.dump(summary_metrics, f, indent=2)

comparison_df.to_csv("notebook5_comparison_table.csv", index=False)
eval_results_df.to_csv("notebook5_eval_results_full.csv", index=False)

print("Saved files:")
print("- notebook5_summary_metrics.json")
print("- notebook5_comparison_table.csv")
print("- notebook5_eval_results_full.csv")



## What we finished

Notebook 5 now covers:
- automated evaluation metrics
- base vs fine-tuned A/B comparison
- row-level evaluation logging
- summary metric logging
- BigQuery evaluation tables
- local evaluation artifacts

Next:
Notebook 6 will prepare deployment.
